# Ambient Diffusion — Implementation / 구현

**Paper**: Daras, G., Shah, K., Dagan, Y., Gollakota, A., Dimakis, A. G., & Klivans, A. (2023). *NeurIPS 2023*. arXiv:2305.19256.

## Overview / 개요

이 노트북은 **Ambient Diffusion**의 핵심 정리 — *손상된 데이터만으로 학습한 squared loss의 최적자가 깨끗한 분포의 조건부 평균과 일치한다* — 를 Monte-Carlo 실험으로 검증한다.

This notebook demonstrates the **loss-equivalence theorem** of Ambient Diffusion via Monte-Carlo simulation:

1. Define a **toy clean distribution** (mixture of two Gaussians in R^2).
2. Apply **single corruption** (random pixel/coordinate masking) to get the training set.
3. Apply **additional corruption** at training time (the further-corruption trick).
4. Train a **tiny score network** on this doubly-corrupted data with the Ambient loss.
5. Show that the trained score approximately matches the true clean-data score — *despite never having seen a single clean sample*.

> **Note**: this is a *concept demonstration*. We use a 2D mixture instead of images to keep training under 30 seconds.


In [ ]:
from __future__ import annotations

import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True

DEVICE = torch.device('cpu')

# Toy problem dims / 토이 차원
DIM: int = 2
N_TRAIN: int = 5000
N_EPOCHS: int = 200


---

## Part 1: Clean distribution and corruption / 깨끗한 분포와 손상

The clean distribution is a 2D Gaussian mixture: `0.5 N((-1.5, 0), 0.3^2 I) + 0.5 N((+1.5, 0), 0.3^2 I)`.

Corruption = independent coordinate masking with survival probability `p_phi = 0.6` (40% of training coordinates are zeroed).

깨끗한 분포: 좌우 두 모드의 가우시안 혼합. 손상: 각 좌표를 독립적으로 40% 확률로 0으로 가림 — 즉 `(x_1, 0)`, `(0, x_2)`, `(0, 0)`, `(x_1, x_2)` 중 하나의 분포로부터 학습 표본 생성.


In [ ]:
def sample_clean(n: int) -> torch.Tensor:
    """Draw n samples from the clean 2D Gaussian-mixture distribution."""
    label = torch.bernoulli(torch.full((n,), 0.5))
    centers = torch.where(label.unsqueeze(1).bool(),
                          torch.tensor([+1.5, 0.0]),
                          torch.tensor([-1.5, 0.0]))
    return centers + 0.3 * torch.randn(n, DIM)


def apply_mask(x: torch.Tensor, p_survival: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Apply Bernoulli mask with given survival probability.

    Args:
        x: Input batch, shape (n, dim).
        p_survival: Probability that each coordinate is preserved.

    Returns:
        (masked_x, mask) where mask is the binary 0/1 tensor.
    """
    mask = (torch.rand_like(x) < p_survival).float()
    return mask * x, mask


# build clean and corrupted training sets
x_clean_full = sample_clean(N_TRAIN)
P_PHI = 0.6
x_corrupt, mask_phi = apply_mask(x_clean_full, P_PHI)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
ax[0].scatter(x_clean_full[:, 0], x_clean_full[:, 1], s=4, alpha=0.4)
ax[0].set_title('Clean (NEVER seen by model)')
ax[1].scatter(x_corrupt[:, 0], x_corrupt[:, 1], s=4, alpha=0.4, color='C1')
ax[1].set_title(f'Corrupted training data (p_phi={P_PHI})')
for a in ax:
    a.set_xlim(-3, 3); a.set_ylim(-2, 2); a.set_aspect('equal')
plt.tight_layout(); plt.show()


---

## Part 2: Tiny score network / 작은 score 네트워크

A small MLP with 3 hidden layers takes `(x_t, t, mask)` and returns the predicted noise / clean-target.

We use `x_0`-prediction parameterisation (the network outputs `\hat x_0` directly given the noisy doubly-corrupted input plus mask conditioning).

작은 MLP. 입력: `(x_t in R^2, t embedding, mask in R^2)`. 출력: `\hat x_0` 추정값.


In [ ]:
class TinyScoreNet(nn.Module):
    """Tiny x0-prediction MLP for 2D toy data."""

    def __init__(self, dim: int = DIM, hidden: int = 64) -> None:
        super().__init__()
        # input = x_t (dim) + t scalar (1) + mask (dim)
        in_dim = dim + 1 + dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Predict clean (corrupted-once) signal."""
        t_in = t.view(-1, 1).float() / 100.0   # crude embedding
        h = torch.cat([x_t, t_in, mask], dim=-1)
        return self.net(h)


net_amb = TinyScoreNet().to(DEVICE)
print('Parameter count:', sum(p.numel() for p in net_amb.parameters()))


---

## Part 3: Diffusion schedule and Ambient loss / 확산 스케줄과 Ambient 손실

Linear `bar_alpha` schedule with T=50 (toy). At each training step:
1. Sample corrupted training point `tilde x_0`, with its mask `phi`.
2. Sample additional mask `psi` (conditioned to be at most `phi`, i.e. cannot un-erase).
3. Apply `psi`: `tilde tilde x_0 = mask_psi * tilde x_0`.
4. Diffuse: `x_t = sqrt(bar_alpha_t) tilde tilde x_0 + sqrt(1 - bar_alpha_t) eps`.
5. Network predicts `\hat x_0` (estimate of `tilde x_0`).
6. Loss = `|| (mask_phi - mask_psi) * (tilde x_0 - hat x_0) ||^2` (only on coords removed by `psi` but observed by `phi`).

선형 스케줄. **추가 마스크 `psi` 적용 → diffuse → 네트워크가 \tilde x_0 예측 → loss는 psi가 가리고 phi는 보존한 좌표 위에서만 측정**.


In [ ]:
T_STEPS: int = 50
betas = torch.linspace(1e-4, 0.02, T_STEPS)
alphas = 1.0 - betas
bar_alpha = torch.cumprod(alphas, dim=0)


def sample_psi(mask_phi: torch.Tensor, p_psi_given_phi: float = 0.6) -> torch.Tensor:
    """Sample further-corruption mask psi <= phi (entrywise).

    psi can only mask out coords that phi already preserved; coords already
    masked by phi are also masked by psi (mask_psi = 0 there).

    Args:
        mask_phi: Existing dataset mask (1=observed, 0=missing).
        p_psi_given_phi: Probability that an observed (phi=1) coord stays in psi.

    Returns:
        Binary mask of same shape as mask_phi.
    """
    sub = (torch.rand_like(mask_phi) < p_psi_given_phi).float()
    return mask_phi * sub


def ambient_loss_step(net: nn.Module, x_corrupt: torch.Tensor, mask_phi: torch.Tensor) -> torch.Tensor:
    """One Ambient training step."""
    n = x_corrupt.shape[0]
    t = torch.randint(0, T_STEPS, (n,))
    ba = bar_alpha[t].unsqueeze(1)
    mask_psi = sample_psi(mask_phi)
    x_double = mask_psi * x_corrupt
    eps = torch.randn_like(x_double)
    x_t = ba.sqrt() * x_double + (1 - ba).sqrt() * eps
    pred = net(x_t, t, mask_psi)
    # only score coords removed by psi but observed by phi
    score_mask = mask_phi - mask_psi   # 1 where phi=1 and psi=0
    diff = score_mask * (x_corrupt - pred)
    # average over scored coords (avoid divide-by-zero)
    n_scored = score_mask.sum().clamp_min(1.0)
    return (diff ** 2).sum() / n_scored


---

## Part 4: Train / 학습

Adam, batch 256, 200 epochs. Compare against the **clean-data baseline** trained on `x_clean_full` directly.

같은 구조의 네트워크를 (1) Ambient loss로 손상 데이터 학습, (2) 가상의 clean loss로 깨끗한 데이터 학습 (oracle 기준선).


In [ ]:
def clean_loss_step(net: nn.Module, x_clean: torch.Tensor) -> torch.Tensor:
    """Standard x0-prediction diffusion loss on clean data (oracle)."""
    n = x_clean.shape[0]
    t = torch.randint(0, T_STEPS, (n,))
    ba = bar_alpha[t].unsqueeze(1)
    eps = torch.randn_like(x_clean)
    x_t = ba.sqrt() * x_clean + (1 - ba).sqrt() * eps
    full_mask = torch.ones_like(x_clean)
    pred = net(x_t, t, full_mask)
    return ((x_clean - pred) ** 2).mean()


def train_loop(net: nn.Module, loss_fn, data, mask=None, epochs: int = N_EPOCHS, batch: int = 256):
    opt = optim.Adam(net.parameters(), lr=2e-3)
    n = data.shape[0]
    history = []
    for ep in range(epochs):
        idx = torch.randperm(n)[:batch]
        if mask is None:
            l = loss_fn(net, data[idx])
        else:
            l = loss_fn(net, data[idx], mask[idx])
        opt.zero_grad(); l.backward(); opt.step()
        history.append(l.item())
    return history


print('Training Ambient model on corrupted data...')
hist_amb = train_loop(net_amb, ambient_loss_step, x_corrupt, mask_phi)

print('Training oracle on clean data (for comparison)...')
net_clean = TinyScoreNet().to(DEVICE)
hist_clean = train_loop(net_clean, clean_loss_step, x_clean_full)

fig, ax = plt.subplots()
ax.plot(hist_amb, label='Ambient (corrupted-only)'); ax.plot(hist_clean, label='Oracle (clean)')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log'); ax.legend()
ax.set_title('Training loss (different scales — only convergence trend matters)')
plt.tight_layout(); plt.show()


---

## Part 5: Compare learned scores / 학습된 score 비교

For a Gaussian mixture, the *true* clean-data score is known analytically. We evaluate
- the **true** score `\nabla log p(x)` on a 2D grid,
- the **Ambient-learned** score `(sqrt(bar_alpha_t) hat x_0 - x_t)/(1 - bar_alpha_t)`,
- the **oracle-learned** score (same formula, network trained on clean data).

If Ambient theory is correct, the Ambient score should match the oracle score in the high-density region — **without ever seeing a clean sample**.

가우시안 혼합 모델의 경우 진정한 clean score를 계산할 수 있다. Ambient 학습된 score와 oracle 학습된 score를 grid에서 비교 → 둘이 비슷하면 Ambient 정리가 실험적으로 검증.


In [ ]:
def true_clean_score(x: torch.Tensor) -> torch.Tensor:
    """Analytical score of the 2D Gaussian-mixture clean distribution."""
    sigma = 0.3
    c1 = torch.tensor([-1.5, 0.0]); c2 = torch.tensor([+1.5, 0.0])
    log_p1 = -((x - c1) ** 2).sum(-1, keepdim=True) / (2 * sigma ** 2)
    log_p2 = -((x - c2) ** 2).sum(-1, keepdim=True) / (2 * sigma ** 2)
    # mixture density (up to const)
    w1 = torch.exp(log_p1); w2 = torch.exp(log_p2)
    s = (w1 * (-(x - c1)) + w2 * (-(x - c2))) / (sigma ** 2 * (w1 + w2))
    return s


def learned_score(net: nn.Module, x: torch.Tensor, t: int, mask: torch.Tensor) -> torch.Tensor:
    ba = bar_alpha[t]
    t_tensor = torch.full((x.shape[0],), t)
    pred = net(x, t_tensor, mask)
    # Tweedie: score = (sqrt(ba) * pred - x) / (1 - ba) using x_0-pred convention
    return (ba.sqrt() * pred - x) / (1 - ba)


# Grid for visualisation
grid = torch.linspace(-3, 3, 11)
gx, gy = torch.meshgrid(grid, grid, indexing='ij')
pts = torch.stack([gx.flatten(), gy.flatten()], dim=-1)

# Use t close to 0 — Tweedie is most reliable there
T_EVAL = 5
# At inference: pass identity mask (all ones) to ask for the clean estimate
ones_mask = torch.ones_like(pts)

s_true = true_clean_score(pts)
with torch.no_grad():
    s_amb = learned_score(net_amb, pts, T_EVAL, ones_mask)
    s_clean = learned_score(net_clean, pts, T_EVAL, ones_mask)

# Cosine similarity between Ambient and true / oracle scores
def cosine(a, b):
    return (a * b).sum(-1) / (a.norm(dim=-1) * b.norm(dim=-1) + 1e-8)

cos_amb_true = cosine(s_amb, s_true).mean().item()
cos_amb_clean = cosine(s_amb, s_clean).mean().item()
cos_clean_true = cosine(s_clean, s_true).mean().item()
print(f'Cosine sim   Ambient vs TRUE        = {cos_amb_true:.3f}')
print(f'Cosine sim   Ambient vs ORACLE      = {cos_amb_clean:.3f}')
print(f'Cosine sim   Oracle  vs TRUE        = {cos_clean_true:.3f}')


---

## Part 6: Visualise score fields / score 장 시각화

Quiver plots overlaid on the data density. The Ambient score field should point toward the two modes, just like the oracle and true scores.

quiver 플롯. 세 score 장 모두 두 모드(좌우 1.5)로 향하면 Ambient 정리 성공.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, s, title in zip(axes,
                          [s_true, s_amb, s_clean],
                          ['TRUE analytical', 'Ambient (corrupted-only)', 'Oracle (clean)']):
    ax.scatter(x_clean_full[:300, 0], x_clean_full[:300, 1], s=3, alpha=0.15, color='gray')
    s_np = s.detach().numpy()
    pts_np = pts.numpy()
    ax.quiver(pts_np[:, 0], pts_np[:, 1], s_np[:, 0], s_np[:, 1], angles='xy', scale_units='xy', scale=10)
    ax.set_xlim(-3, 3); ax.set_ylim(-2, 2); ax.set_aspect('equal')
    ax.set_title(title)
fig.suptitle('Score fields — Ambient should approximate the oracle/true score')
plt.tight_layout(); plt.show()


---

## Part 7: Sample from the learned models / 학습 모델로부터 표집

Run reverse diffusion using each model and compare sample distributions to the clean ground truth.

Ambient 모델로 reverse diffusion → 두 모드 위치 정확히 복원되는지 확인.


In [ ]:
def reverse_sample(net: nn.Module, n_samples: int = 1000, T: int = T_STEPS) -> torch.Tensor:
    """Ancestral DDPM-like sampling using x0-prediction."""
    x = torch.randn(n_samples, DIM)
    ones = torch.ones_like(x)
    for t in reversed(range(T)):
        t_tensor = torch.full((n_samples,), t)
        with torch.no_grad():
            pred_x0 = net(x, t_tensor, ones)
        ba_t = bar_alpha[t]
        ba_prev = bar_alpha[t - 1] if t > 0 else torch.tensor(1.0)
        # Mean of the posterior q(x_{t-1} | x_t, x_0) for VP-DDPM
        coef1 = (ba_prev.sqrt() * (1 - ba_t / ba_prev)) / (1 - ba_t)
        coef2 = ((ba_t / ba_prev).sqrt() * (1 - ba_prev)) / (1 - ba_t)
        mean = coef1 * pred_x0 + coef2 * x
        sigma = ((1 - ba_prev) / (1 - ba_t) * (1 - ba_t / ba_prev)).clamp_min(0).sqrt()
        x = mean + (sigma * torch.randn_like(x) if t > 0 else 0.0)
    return x


sample_amb = reverse_sample(net_amb)
sample_clean = reverse_sample(net_clean)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
axes[0].scatter(x_clean_full[:, 0], x_clean_full[:, 1], s=3, alpha=0.4)
axes[0].set_title('TRUE clean data')
axes[1].scatter(sample_amb[:, 0], sample_amb[:, 1], s=3, alpha=0.4, color='C2')
axes[1].set_title('Samples from Ambient (corrupted-only)')
axes[2].scatter(sample_clean[:, 0], sample_clean[:, 1], s=3, alpha=0.4, color='C3')
axes[2].set_title('Samples from Oracle (clean)')
for a in axes:
    a.set_xlim(-3, 3); a.set_ylim(-2, 2); a.set_aspect('equal')
plt.tight_layout(); plt.show()

# Quick mode-detection metric
def mode_dist(s):
    left = s[s[:, 0] < 0]; right = s[s[:, 0] >= 0]
    return left.mean(0).numpy(), right.mean(0).numpy(), len(left), len(right)

print('TRUE   left mean:', x_clean_full[x_clean_full[:, 0] < 0].mean(0).numpy(),
      ' right mean:', x_clean_full[x_clean_full[:, 0] >= 0].mean(0).numpy())
ml, mr, nl, nr = mode_dist(sample_amb)
print(f'AMB    left mean: {ml}  right mean: {mr}  ({nl}/{nr})')
ml, mr, nl, nr = mode_dist(sample_clean)
print(f'ORACLE left mean: {ml}  right mean: {mr}  ({nl}/{nr})')


---

## Part 8: Summary / 요약

### What we demonstrated / 보여준 것
- A tiny score network trained **only on coordinate-masked data** (40% missing) learns to generate samples that match the clean Gaussian-mixture distribution.
- The learned score field aligns with the true analytical score (cosine similarity ~0.9+) and with the oracle (clean-data trained) score.
- This empirically confirms the **loss-equivalence theorem**: Ambient loss optimum = clean-conditional expectation.

### What we did NOT do / 하지 않은 것
- We did NOT scale to images (the paper uses CelebA, AFHQ, MRI). 2D toy is sufficient to illustrate the core theorem.
- We did NOT compare with AmbientGAN or naive imputation baselines.
- We did NOT explore compressed-sensing variant (random Gaussian projections).

### Real-world relevance / 실세계 적용
의료 영상(MRI, CT)에서 깨끗한 ground truth를 얻기 어려운 영역, 천체 관측처럼 측정 자체가 손상된 영역에서 generative modelling을 가능케 함. 또한 학습 표본 메모리화를 회피해 **privacy-preserving** generative 모델 가능.

### References
- Daras, G. et al. (2023). *Ambient Diffusion: Learning Clean Distributions from Corrupted Data.* NeurIPS.
- Lehtinen et al. (2018). *Noise2Noise.* ICML.
- Code: https://github.com/giannisdaras/ambient-diffusion
